In [ ]:
tmp_figdir = '/Users/ahoffman/personal_gkyl_scripts/studies/tcv_miller_scan_scripts/figures/'
manuscript_figdir = '/Users/ahoffman/writing/gkyl_tcv_miller_scan/figures/'
h5file = 'data/tcv_miller_scan_big_metadata_frame_500_navg_25_copy.h5'

from scan_analysis.scan_metadata import ScanMetadata
miller_scan = ScanMetadata(h5file)


root_run_dir = '/Users/ahoffman/personal_gkyl_scripts/studies/tcv_miller_scan_scripts/gyacomo_run_data/'
std_param_file = '/Users/ahoffman/personal_gkyl_scripts/studies/tcv_miller_scan_scripts/gyacomo_input_template.F90'
gyac_exe = '/Users/ahoffman/gyacomo/bin/gyacomo23_dp'

ky = 0.5
gyac_params = {
    'q0': 2.6,
    'shear': 1.9,
    'eps': 0.28,
    'Lx': 200.0,
    'Ly': 2*3.14/ky,
    'Nx': 16,
    'Ny': 2,
    'Nz': 24,
    'pmax': 4,
    'jmax': 2,
    'tmax': 1.0,
    'dt': 1e-3,
    'nu': 1.0
}

In [ ]:
# get data from the Gkeyll simulations
import numpy as np
from concurrent.futures import ThreadPoolExecutor, as_completed
from gyacomo_interface.gyacomo_simulation import GyacomoSimulation

kappa_vals = miller_scan.scan_params['kappa']
delta_vals = miller_scan.scan_params['delta']
energy_srcCORE_vals = miller_scan.scan_params['energy_srcCORE']

gyac_linear_results = {}

def run_gyac_kernel(params, verbose=False):
    # Get parameters from Gkeyll scan
    p_ = {'kappa': params[0], 'delta': params[1], 'energy_srcCORE': params[2]}
    tau = miller_scan.get_field_value('tau_lcfs', params=p_)
    kTe = miller_scan.get_field_value('kTe', params=p_)
    kTi = miller_scan.get_field_value('kTi', params=p_)
    kne = miller_scan.get_field_value('kne', params=p_)
    tau = miller_scan.get_field_value('tau_lcfs', params=p_)
    
    # Initialize simulation
    run_dir = f"{root_run_dir}/kappa_{p_['kappa']:.2f}_delta_{p_['delta']:.2f}_energy_{p_['energy_srcCORE']:.1e}"
    sim = GyacomoSimulation(run_dir=run_dir,jobnum=0,executable=gyac_exe)
    sim.load_params_from_file(std_param_file)
    
    # Set up the constant parameters
    sim.set_param(gyac_params)
    
    # Set up the scan parameters
    sim.set_param({'delta': p_['delta']})
    sim.set_param({'kappa': p_['kappa']})
    sim.set_param({'k_N_e': kne})
    sim.set_param({'k_T_e': kTe})
    sim.set_param({'k_N_i': kne})
    sim.set_param({'k_T_i': kTi})
    sim.set_param({'tau_e': tau})

    # Setup directory and write params
    sim.setup_directory(clean=True)
    
    if verbose:
        sidx = miller_scan.get_sim_index(p_)
        print(f"i={sidx:05d}, d={p_['delta']:1.1f}, k={p_['kappa']:1.1f}, e={p_['energy_srcCORE']:1.1e}, tau={tau:1.1f}, kTe={kTe:2.1f}, kTi={kTi:2.1f}, kne={kne:2.1f}")

    # Run the simulation
    _ = sim.run(nproc=4, proc_distr=(1, 2, 2), blocking=True, verbose=False)

    # Compute growth rate
    gr_dict = sim.compute_growth_rate('phi', kx_val=0, ky_index=1)
    ky = gr_dict['ky'][0]
    omega = gr_dict['omega'][0]
    
    if verbose:
        print(f"    ky={ky:2.1f}, Ɣ={np.real(omega):2.1f}, ω={np.imag(omega):2.1f}")
    
    return ky, omega, sim.get_params_content()


def _run_with_index(args):
    i, params = args
    ky, omega, param_content = run_gyac_kernel(params, verbose=True)
    return i, ky, omega, param_content


n_workers = 4  # number of parallel simulations

ky_array = np.zeros(len(miller_scan.combinations))
omega_array = np.zeros(len(miller_scan.combinations), dtype=complex)
params_in_array = np.empty(len(miller_scan.combinations), dtype=object)

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    futures = {
        executor.submit(_run_with_index, (i, params)): i
        for i, params in enumerate(miller_scan.combinations)
    }
    for future in as_completed(futures):
        i, ky, omega, param_content = future.result()
        ky_array[i] = ky
        omega_array[i] = omega
        params_in_array[i] = param_content


In [ ]:
miller_scan.save_gk_linear_data()